In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

In [4]:
import os
import kagglehub
path = kagglehub.dataset_download("rabieelkharoua/predict-conversion-in-digital-marketing-dataset")

print(path)
print(os.listdir(path))
import pandas as pd
import os

df_m = pd.read_csv(os.path.join(path, "digital_marketing_campaign_dataset.csv"))
df_m.head()


/Users/riverocel/.cache/kagglehub/datasets/rabieelkharoua/predict-conversion-in-digital-marketing-dataset/versions/1
['digital_marketing_campaign_dataset.csv']


,CustomerID,Age,Gender,Income,CampaignChannel,CampaignType,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,AdvertisingPlatform,AdvertisingTool,Conversion
0,8000,56,Female,136912,Social Media,Awareness,6497.870068,0.043919,0.088031,0,2.399017,7.396803,19,6,9,4,688,IsConfid,ToolConfid,1
1,8001,69,Male,41760,Email,Retention,3898.668606,0.155725,0.182725,42,2.917138,5.352549,5,2,7,2,3459,IsConfid,ToolConfid,1
2,8002,46,Female,88456,PPC,Awareness,1546.429596,0.277490,0.076423,2,8.223619,13.794901,0,11,2,8,2337,IsConfid,ToolConfid,1
3,8003,32,Female,44085,PPC,Conversion,539.525936,0.137611,0.088004,47,4.540939,14.688363,89,2,2,0,2463,IsConfid,ToolConfid,1
4,8004,60,Female,83964,PPC,Conversion,1678.043573,0.252851,0.109940,0,2.046847,13.993370,6,6,6,8,4345,IsConfid,ToolConfid,1


In [7]:
target = "Conversion"

features = [
    "Age",
    "Income",
    "AdSpend",
    "ClickThroughRate",
    "WebsiteVisits",
    "PagesPerVisit",
    "TimeOnSite",
    "EmailOpens",
    "EmailClicks",
    "PreviousPurchases",
    "LoyaltyPoints"
]

data = df_m[[target] + features].dropna()
X = data[features]
y = data[target]

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
log_model = LogisticRegression(max_iter=500)

log_model.fit(X_train_scaled, y_train)

y_pred_log = log_model.predict(X_test_scaled)

print("LOGISTIC REGRESSION")

print("Accuracy:",
      accuracy_score(y_test, y_pred_log))

print("ROC AUC:",
      roc_auc_score(y_test, y_pred_log))

print(confusion_matrix(y_test, y_pred_log))

print(classification_report(y_test, y_pred_log))

LOGISTIC REGRESSION
Accuracy: 0.88125
ROC AUC: 0.5527313073675413
[[  23  175]
 [  15 1387]]
              precision    recall  f1-score   support

           0       0.61      0.12      0.19       198
           1       0.89      0.99      0.94      1402

    accuracy                           0.88      1600
   macro avg       0.75      0.55      0.57      1600
weighted avg       0.85      0.88      0.84      1600



for scaled logistic regression on how many people converted, the model is favoring the converted data because it is not balanced.  And we can see that with the recall difference between non converted near 0% accuracy on finding true non converted usage. The ROC curve is just eh anyways.

In [26]:
lasso_model = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C=0.01,
    class_weight='balanced',
    max_iter=1000
)

lasso_model.fit(X_train_scaled, y_train)

print(lasso_model.coef_)

[[0.         0.         0.28485671 0.34468306 0.14134752 0.209576
  0.34297126 0.26726932 0.31768192 0.26769305 0.2490749 ]]


In [27]:
ridge_model = LogisticRegression(
    penalty='l2',
    solver='liblinear',
    C=0.01,
    class_weight='balanced',
    max_iter=1000
)

ridge_model.fit(X_train_scaled, y_train)

LogisticRegression(C=0.01, class_weight='balanced', max_iter=1000,
                   solver='liblinear')

In [28]:
elastic_model = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    l1_ratio=0.5,
    C=0.01,
    class_weight='balanced',
    max_iter=5000
)

elastic_model.fit(X_train_scaled, y_train)

LogisticRegression(C=0.01, class_weight='balanced', l1_ratio=0.5, max_iter=5000,
                   penalty='elasticnet', solver='saga')

In [29]:
coef_df = pd.DataFrame({
    "Feature": features,
    "Logistic": log_model.coef_[0],
    "Ridge": ridge_model.coef_[0],
    "Lasso": lasso_model.coef_[0],
    "ElasticNet": elastic_model.coef_[0]
})

print(coef_df.round(4))

              Feature  Logistic   Ridge   Lasso  ElasticNet
0                 Age   -0.0140 -0.0062  0.0000      0.0000
1              Income    0.0418  0.0450  0.0000      0.0088
2             AdSpend    0.4361  0.3456  0.2849      0.3232
3    ClickThroughRate    0.4954  0.4136  0.3447      0.3891
4       WebsiteVisits    0.2780  0.2061  0.1413      0.1782
5       PagesPerVisit    0.3715  0.2683  0.2096      0.2444
6          TimeOnSite    0.5074  0.4073  0.3430      0.3850
7          EmailOpens    0.4198  0.3287  0.2673      0.3058
8         EmailClicks    0.4799  0.3823  0.3177      0.3592
9   PreviousPurchases    0.4234  0.3275  0.2677      0.3047
10      LoyaltyPoints    0.3834  0.3179  0.2491      0.2909


largest coefficients are result of logistic regression as we expected as it's the base model, Lasso decreased age and income to 0 shows those variables were not significant in demographic data using feature selection. Elastic net being a mix of both stabilized the variables and shrank them. The strongest variables seemed with AD spend, time on site, email clicks and previous purchases.